# Bölüm 5: İstatistiksel Çıkarım Temelleri

**Haydar Kılıç — Yapay Zeka Mühendisliği**

Bu notebook, istatistiksel çıkarımın temel kavramlarını Python ile uygulamalı olarak ele almaktadır:

1. Nokta Tahminleri ve Örnekleme Değişkenliği
2. Örnekleme Dağılımı
3. Merkezi Limit Teoremi (MLT)
4. Güven Aralıkları
5. Hipotez Testi
6. Karar Hataları (Tip 1 / Tip 2)

---

In [ ]:
# Gerekli kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(42)
print("Kütüphaneler başarıyla yüklendi.")

---
## 1. Nokta Tahminleri ve Örnekleme Değişkenliği

**Teori:**
- Kitle parametrelerini (örn. ortalama, oran) doğrudan ölçemeyiz.
- Örneklemden hesaplanan istatistikleri **nokta tahmini** olarak kullanırız.
- **Yanlılık:** Tahmin edilen değerin sistematik olarak sapması.
- **Örnekleme hatası:** Tahminlerin örneklemden örnekleme değişimi.

### 1.1 Örnek: ABD'de Boy Ortalaması

Her eyaletten 1000 yetişkin örneklenseydi, ortalama boylar birbirine çok yakın ama aynı olmayacaktı.

In [ ]:
# Gerçek kitle: ortalama 175 cm, std 10 cm
KITLE_ORTALAMA = 175
KITLE_STD = 10
ORNEK_BUYUKLUGU = 1000
N_EYALET = 50

# Her eyalet için örneklem ortalaması simüle et
eyalet_ortalamalar = [
    np.mean(np.random.normal(KITLE_ORTALAMA, KITLE_STD, ORNEK_BUYUKLUGU))
    for _ in range(N_EYALET)
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(1, N_EYALET + 1), eyalet_ortalamalar,
       color='steelblue', edgecolor='white', alpha=0.85)
ax.axhline(KITLE_ORTALAMA, color='red', linestyle='--',
           linewidth=2, label=f'Gerçek kitle ortalaması = {KITLE_ORTALAMA} cm')
ax.set_xlabel('Eyalet')
ax.set_ylabel('Örneklem Ortalaması (cm)')
ax.set_title('50 Eyaletten Elde Edilen Boy Ortalamaları (n=1000)')
ax.set_ylim(172, 178)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Örneklem ortalamalarının aralığı: [{min(eyalet_ortalamalar):.2f}, {max(eyalet_ortalamalar):.2f}]")
print(f"Gerçek kitle ortalaması: {KITLE_ORTALAMA} cm")
print("→ Tüm tahminler birbirine çok yakın ama tamamen aynı değil!")

---
## 2. Örnekleme Dağılımı

**Teori:**
Aynı prosedürü çok sayıda tekrarladığımızda elde ettiğimiz $\hat{p}$ değerlerinin oluşturduğu dağılıma **örnekleme dağılımı** denir.

> Gerçek dünyada örnekleme dağılımını hiçbir zaman gözlemleyemeyiz; ancak her nokta tahminini bu varsayımsal dağılımdan geliyormuş gibi düşünmek çok yararlıdır.

### 2.1 Güneş Enerjisi Örneği (Sunumdaki Simülasyon)

Gerçek kitle oranı **p = 0.88** (güneş enerjisini destekleyen Amerikalılar).

In [ ]:
# Kitle parametresi
p_gercek = 0.88
n = 1000          # örneklem büyüklüğü
n_tekrar = 10000  # kaç kez örneklem alınıyor

# Her seferinde örneklem al ve p-hat hesapla
p_hat_listesi = np.random.binomial(n, p_gercek, n_tekrar) / n

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(p_hat_listesi, bins=30, color='steelblue',
        edgecolor='white', alpha=0.85, density=False)
ax.axvline(p_gercek, color='red', linestyle='--',
           linewidth=2.5, label=f'Gerçek p = {p_gercek}')
ax.axvline(np.mean(p_hat_listesi), color='orange', linestyle='-.',
           linewidth=2, label=f'Örnekleme ort. = {np.mean(p_hat_listesi):.3f}')
ax.set_xlabel('Simüle edilmiş örneklem oranı ($\hat{p}$)')
ax.set_ylabel('Frekans')
ax.set_title('Örnekleme Dağılımı (p=0.88, n=1000, 10.000 tekrar)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Örnekleme dağılımının ortalaması : {np.mean(p_hat_listesi):.4f}")
print(f"Örnekleme dağılımının std sapması : {np.std(p_hat_listesi):.4f}")
print(f"Teorik standart hata             : {np.sqrt(p_gercek*(1-p_gercek)/n):.4f}")

---
## 3. Merkezi Limit Teoremi (MLT)

**Teori:**
$$\hat{p} \sim N\left(p,\ SE = \sqrt{\frac{p(1-p)}{n}}\right)$$

**MLT Koşulları:**
1. **Bağımsızlık:** Gözlemler bağımsız olmalı (rastgele örnekleme + n < nüfusun %10'u)
2. **Örneklem büyüklüğü:** $np \geq 10$ **ve** $n(1-p) \geq 10$

### 3.1 n ve p'nin Dağılım Şekline Etkisi

In [ ]:
p_degerleri = [0.1, 0.2, 0.5, 0.8, 0.9]
n_degerleri = [10, 25, 50, 100, 250]
n_sim = 3000

fig, axes = plt.subplots(len(p_degerleri), len(n_degerleri),
                         figsize=(16, 12), sharex=True)

for i, p in enumerate(p_degerleri):
    for j, n_val in enumerate(n_degerleri):
        simulasyon = np.random.binomial(n_val, p, n_sim) / n_val
        axes[i, j].hist(simulasyon, bins=20, color='steelblue',
                        edgecolor='none', alpha=0.8, density=True)
        axes[i, j].set_xlim(0, 1)
        axes[i, j].set_yticks([])
        axes[i, j].tick_params(labelsize=7)

        # Koşul kontrolü
        kosul = (n_val * p >= 10) and (n_val * (1 - p) >= 10)
        renk = 'green' if kosul else 'red'
        axes[i, j].set_title(f'n={n_val}', fontsize=8, color='black')
        for sp in axes[i, j].spines.values():
            sp.set_edgecolor(renk)
            sp.set_linewidth(2)

        if j == 0:
            axes[i, j].set_ylabel(f'p={p}', fontsize=9)

# Legend
yesil = mpatches.Patch(color='green', label='MLT koşulu sağlanıyor (np≥10 ve n(1-p)≥10)')
kirmizi = mpatches.Patch(color='red', label='MLT koşulu SAĞLANMIYOR')
fig.legend(handles=[yesil, kirmizi], loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))

fig.suptitle('np ve n(1-p) < 10 Olduğunda Örnekleme Dağılımının Şekli',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── MLT Koşul Kontrolü – Yardımcı Fonksiyon ──────────────────────────
def mlt_kosul_kontrol(n, p=None, p_hat=None, nufus_buyuklugu=None):
    """
    MLT koşullarını kontrol eder.
    p bilinmiyorsa p_hat kullanılır.
    """
    p_kullan = p if p is not None else p_hat
    if p_kullan is None:
        raise ValueError("p veya p_hat girilmelidir.")

    print("=" * 50)
    print("MLT Koşul Kontrolü")
    print("=" * 50)

    # Koşul 1: Bağımsızlık / %10 kuralı
    if nufus_buyuklugu:
        oran = n / nufus_buyuklugu
        k1 = oran < 0.10
        print(f"1) %10 kuralı: n/N = {oran:.4f} → {'✓ SAĞLANIYOR' if k1 else '✗ SAĞLANMIYOR'}")
    else:
        print("1) %10 kuralı: Nüfus büyüklüğü bilinmiyor, rastgele örnekleme varsayılıyor.")
        k1 = True

    # Koşul 2: Başarı-başarısızlık
    np_val = n * p_kullan
    n1p_val = n * (1 - p_kullan)
    k2 = (np_val >= 10) and (n1p_val >= 10)
    print(f"2) Başarı-başarısızlık:")
    print(f"   np  = {n} × {p_kullan:.3f} = {np_val:.1f}  → {'✓' if np_val >= 10 else '✗ (< 10!)'}")
    print(f"   n(1-p) = {n} × {1-p_kullan:.3f} = {n1p_val:.1f}  → {'✓' if n1p_val >= 10 else '✗ (< 10!)'}")

    sonuc = k1 and k2
    print("-" * 50)
    print(f"SONUÇ: MLT {'UYGULANAB İLİR ✓' if sonuc else 'UYGULANAMAZ ✗'}")
    print("=" * 50)
    return sonuc

# Güneş enerjisi örneği
mlt_kosul_kontrol(n=1000, p=0.88, nufus_buyuklugu=250_000_000)

In [ ]:
# p=0.05, n=50 → koşul SAĞLANMIYOR
print("Sunumdaki örnek: p=0.05, n=50")
mlt_kosul_kontrol(n=50, p=0.05)

# Görselleştirme
p_dusuk, n_kucuk = 0.05, 50
sim_dusuk = np.random.binomial(n_kucuk, p_dusuk, 5000) / n_kucuk

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sim_dusuk, bins=15, color='tomato', edgecolor='white', alpha=0.85)
ax.set_xlabel('Örneklem oranı ($\hat{p}$)')
ax.set_ylabel('Frekans')
ax.set_title(f'p={p_dusuk}, n={n_kucuk} → Çarpık dağılım (koşul sağlanmıyor!)')
ax.text(0.10, 200, f'np = {n_kucuk*p_dusuk:.1f} < 10\n→ Normal yaklaşım geçersiz!',
        fontsize=11, color='darkred',
        bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.8))
plt.tight_layout()
plt.show()

---
## 4. Güven Aralıkları

**Teori:**
$$\hat{p} \pm z^* \times SE, \qquad SE = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

| Güven Düzeyi | z* |
|:---:|:---:|
| %90 | 1.645 |
| %95 | 1.960 |
| %98 | 2.326 |
| %99 | 2.576 |

### 4.1 Facebook İlgi Alanları Örneği (Sunumdaki Hesaplama)

In [ ]:
def guven_araligi(p_hat, n, guven=0.95):
    """
    Bir oran için güven aralığı hesaplar.
    """
    alpha = 1 - guven
    z_star = stats.norm.ppf(1 - alpha / 2)
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    hata_payi = z_star * se
    alt = p_hat - hata_payi
    ust = p_hat + hata_payi

    print(f"{'='*55}")
    print(f"Güven Aralığı Hesabı ({guven*100:.0f}%)")
    print(f"{'='*55}")
    print(f"  p̂ (örneklem oranı) : {p_hat}")
    print(f"  n (örneklem büyük.): {n}")
    print(f"  z* (kritik değer)  : {z_star:.4f}")
    print(f"  SE                 : √({p_hat}×{1-p_hat}/{n}) = {se:.4f}")
    print(f"  Hata payı          : {z_star:.4f} × {se:.4f} = {hata_payi:.4f} ≈ {hata_payi:.2f}")
    print(f"-{'-'*54}")
    print(f"  Güven aralığı      : ({alt:.4f}, {ust:.4f})"
          f"  ≈  (%{alt*100:.1f}, %{ust*100:.1f})")
    print(f"{'='*55}")
    return alt, ust, z_star, se

# Sunumdaki örnek
alt, ust, _, _ = guven_araligi(p_hat=0.67, n=850, guven=0.95)

In [ ]:
# Farklı güven düzeyleri karşılaştırması
p_hat, n = 0.67, 850
guven_duzeyleri = [0.90, 0.95, 0.98, 0.99]
renkler = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

fig, ax = plt.subplots(figsize=(11, 5))

for i, (guven, renk) in enumerate(zip(guven_duzeyleri, renkler)):
    z = stats.norm.ppf(1 - (1 - guven) / 2)
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    alt_val = p_hat - z * se
    ust_val = p_hat + z * se

    ax.plot([alt_val, ust_val], [i, i], color=renk, linewidth=5, solid_capstyle='round')
    ax.plot(p_hat, i, 'o', color='black', markersize=7, zorder=5)
    ax.text(ust_val + 0.003, i, f'%{guven*100:.0f}  [{alt_val:.3f}, {ust_val:.3f}]',
            va='center', fontsize=10, color=renk)

ax.axvline(p_hat, color='black', linestyle='--', alpha=0.4, label=f'p̂ = {p_hat}')
ax.set_yticks(range(len(guven_duzeyleri)))
ax.set_yticklabels([f'%{int(g*100)}' for g in guven_duzeyleri])
ax.set_xlabel('Oran')
ax.set_title('Farklı Güven Düzeylerinde Güven Aralıkları (p̂=0.67, n=850)')
ax.set_xlim(0.60, 0.78)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()
print("Güven düzeyi arttıkça aralık GENİŞLER → daha az kesin bilgi verir!")

### 4.2 '%95 Güven' Ne Anlama Gelir?

Çok sayıda örneklem aldığımızda, oluşturulan aralıkların yaklaşık **%95'i** gerçek kitle parametresini içerir.

In [ ]:
# %95 guven anlaminin simulasyonu
p_gercek = 0.67
n = 850
n_orneklem = 100  # kac farkli orneklem
guven = 0.95
z_star = stats.norm.ppf(1 - (1 - guven) / 2)

kapsayan_sayisi = 0
fig, ax = plt.subplots(figsize=(12, 8))

for i in range(n_orneklem):
    phat = np.random.binomial(n, p_gercek) / n
    se = np.sqrt(phat * (1 - phat) / n)
    alt_val = phat - z_star * se
    ust_val = phat + z_star * se
    kapsiyor = alt_val <= p_gercek <= ust_val

    if kapsiyor:
        kapsayan_sayisi += 1

    renk = "#4CAF50" if kapsiyor else "#F44336"
    ax.plot([alt_val, ust_val], [i, i], color=renk, linewidth=1.5,
            alpha=0.75, solid_capstyle="round")

ax.axvline(p_gercek, color="black", linewidth=2.5,
           linestyle="--", label=f"Gercek p = {p_gercek}")

yesil_p = mpatches.Patch(color="#4CAF50", label=f"Kapsiyor ({kapsayan_sayisi}/{n_orneklem})")
kirmizi_p = mpatches.Patch(color="#F44336", label=f"Kapsamiyor ({n_orneklem - kapsayan_sayisi}/{n_orneklem})")
ax.legend(handles=[yesil_p, kirmizi_p,
                   mpatches.Patch(color="black", label=f"Gercek p={p_gercek}")],
          fontsize=10)

ax.set_xlabel("Oran")
ax.set_ylabel("Orneklem")
ax.set_title(f"%95 Guven Araligi: 100 Orneklemden {kapsayan_sayisi} tanesi gercek p yi kapsiyor")
ax.set_yticks([])
plt.tight_layout()
plt.show()

print("Kapsama orani:", kapsayan_sayisi, "/", n_orneklem, "= %", kapsayan_sayisi)
print("Teorik beklenti: %95")

---
## 5. Hipotez Testi

**Çerçeve:**
- $H_0$: Sıfır hipotezi (statüko)
- $H_A$: Alternatif hipotez (araştırma sorusu)

$$Z = \frac{\hat{p} - p_0}{SE_0}, \qquad SE_0 = \sqrt{\frac{p_0(1-p_0)}{n}}$$

### 5.1 Facebook İlgi Kategorileri Hipotez Testi (Sunumdaki Örnek)

$H_0: p = 0.50$ vs $H_A: p \neq 0.50$, $\hat{p} = 0.41$, $n = 850$

In [ ]:
def hipotez_testi_oran(p_hat, n, p_0, yon='iki_yonlu', alpha=0.05):
    """
    Bir oran için z-testi.
    yon: 'iki_yonlu', 'sol', 'sag'
    """
    se_0 = np.sqrt(p_0 * (1 - p_0) / n)
    z = (p_hat - p_0) / se_0

    if yon == 'iki_yonlu':
        p_deger = 2 * stats.norm.sf(abs(z))
        h_a = f'p ≠ {p_0}'
    elif yon == 'sol':
        p_deger = stats.norm.cdf(z)
        h_a = f'p < {p_0}'
    else:
        p_deger = stats.norm.sf(z)
        h_a = f'p > {p_0}'

    print(f"{'='*55}")
    print("Hipotez Testi")
    print(f"{'='*55}")
    print(f"  H₀ : p = {p_0}")
    print(f"  Hₐ : {h_a}")
    print(f"  p̂  = {p_hat},  n = {n},  α = {alpha}")
    print("-" * 55)
    print(f"  SE₀ = √({p_0}×{1-p_0}/{n}) = {se_0:.4f}")
    print(f"  Z   = ({p_hat} - {p_0}) / {se_0:.4f} = {z:.4f}")
    print(f"  p-değeri = {p_deger:.6f}")
    print("-" * 55)
    if p_deger < alpha:
        print(f"  KARAR: p-değeri ({p_deger:.4f}) < α ({alpha})")
        print(f"  → H₀ REDDEDİLİR. Hₐ lehine yeterli kanıt var.")
    else:
        print(f"  KARAR: p-değeri ({p_deger:.4f}) ≥ α ({alpha})")
        print(f"  → H₀ REDDEDİLEMEZ. Hₐ için yeterli kanıt yok.")
    print(f"{'='*55}")
    return z, p_deger

# Sunumdaki örnek
z_istat, p_deger = hipotez_testi_oran(p_hat=0.41, n=850, p_0=0.50, yon='iki_yonlu')

In [ ]:
# p-değeri görselleştirmesi
fig, ax = plt.subplots(figsize=(11, 5))

x = np.linspace(-7, 7, 1000)
y = stats.norm.pdf(x)

ax.plot(x, y, 'k-', linewidth=2)

# İki kuyruk: |Z| > 5.26
z_krit = abs(z_istat)
x_sol = x[x <= -z_krit]
x_sag = x[x >= z_krit]
ax.fill_between(x_sol, stats.norm.pdf(x_sol), color='tomato', alpha=0.7,
                label=f'p-değeri < 0.0001')
ax.fill_between(x_sag, stats.norm.pdf(x_sag), color='tomato', alpha=0.7)

# α = 0.05 kritik değerleri
z_alpha = stats.norm.ppf(0.975)
ax.axvline(-z_alpha, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'α=0.05 kritik değer: ±{z_alpha:.2f}')
ax.axvline(z_alpha, color='steelblue', linestyle='--', linewidth=1.5)

ax.axvline(z_istat, color='darkred', linewidth=2.5,
           label=f'Test istatistiği Z = {z_istat:.2f}')

ax.set_xlabel('Z')
ax.set_ylabel('Yoğunluk')
ax.set_title('Hipotez Testi: p-değeri Görselleştirmesi\n'
             'H₀: p=0.50  vs  Hₐ: p≠0.50  (p̂=0.41, n=850)')
ax.legend(fontsize=10)
ax.set_xlim(-7, 7)
plt.tight_layout()
plt.show()

### 5.2 Tek Yönlü vs İki Yönlü Test

**Sunumdaki Facebook güven aralığı örneği:**  
Güven aralığı (%64, %70) → $p_0 = 0.50$ aralığa **girmediğinden** $H_0$ reddedilir.

In [ ]:
def guven_araligi_ile_test(p_hat, n, p_0, guven=0.95):
    """Güven aralığı kullanarak hipotez testi."""
    alt_val, ust_val, _, _ = guven_araligi(p_hat, n, guven)
    print(f"\nH₀: p = {p_0}")
    print(f"Sıfır değeri ({p_0}) aralıkta mı? ", end="")
    if alt_val <= p_0 <= ust_val:
        print(f"EVET → H₀ reddedilemez.")
    else:
        print(f"HAYIR → H₀ REDDEDİLİR!")

guven_araligi_ile_test(p_hat=0.67, n=850, p_0=0.50, guven=0.95)

# Görsel
fig, ax = plt.subplots(figsize=(9, 3))
z_star = stats.norm.ppf(0.975)
se = np.sqrt(0.67 * 0.33 / 850)
alt_val2 = 0.67 - z_star * se
ust_val2 = 0.67 + z_star * se

ax.plot([alt_val2, ust_val2], [0, 0], color='steelblue', linewidth=10,
        solid_capstyle='round', label=f'%95 GA: [{alt_val2:.3f}, {ust_val2:.3f}]')
ax.plot(0.67, 0, 'o', color='black', markersize=10, zorder=5, label='p̂ = 0.67')
ax.axvline(0.50, color='red', linewidth=2.5, linestyle='--',
           label='H₀: p₀ = 0.50')
ax.set_yticks([])
ax.set_xlabel('Oran')
ax.set_title('Güven Aralığı ile Hipotez Testi\np₀=0.50 aralık dışında → H₀ reddedilir!')
ax.legend(fontsize=10)
ax.set_xlim(0.44, 0.78)
plt.tight_layout()
plt.show()

---
## 6. Karar Hataları

| | H₀ reddedilmez | H₀ reddedilir |
|:---|:---:|:---:|
| **H₀ doğru** | ✓ Doğru karar | ✗ Tip 1 Hata (α) |
| **Hₐ doğru** | ✗ Tip 2 Hata (β) | ✓ Doğru karar |

- **Tip 1 Hata:** H₀ doğruyken reddetmek (yanlış alarm)
- **Tip 2 Hata:** Hₐ doğruyken H₀'ı reddetmemek (gözden kaçırma)

$$P(\text{Tip 1 Hata} \mid H_0 \text{ doğru}) = \alpha$$

In [ ]:
# Karar hataları görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(-5, 8, 1000)
mu_0, mu_a = 0, 3  # H₀ ve Hₐ dağılım ortalamaları
sigma = 1
z_krit = stats.norm.ppf(0.95)  # tek yönlü α=0.05

y0 = stats.norm.pdf(x, mu_0, sigma)
ya = stats.norm.pdf(x, mu_a, sigma)

# Sol panel: Tip 1 Hata
ax = axes[0]
ax.plot(x, y0, 'steelblue', linewidth=2.5, label='H₀ dağılımı')
x_tip1 = x[x >= z_krit]
ax.fill_between(x_tip1, stats.norm.pdf(x_tip1, mu_0, sigma),
                color='tomato', alpha=0.7, label=f'Tip 1 Hata (α={0.05})')
ax.axvline(z_krit, color='black', linestyle='--', linewidth=1.5)
ax.set_title('Tip 1 Hata\nH₀ doğruyken H₀\'ı reddetmek')
ax.set_xlabel('Test istatistiği')
ax.legend(fontsize=9)
ax.set_yticks([])
ax.text(z_krit + 0.1, 0.15, 'Red\nbölgesi', fontsize=9)

# Sağ panel: Tip 2 Hata
ax = axes[1]
ax.plot(x, y0, 'steelblue', linewidth=2.5, label='H₀ dağılımı', alpha=0.6)
ax.plot(x, ya, 'darkorange', linewidth=2.5, label='Hₐ dağılımı')
x_tip2 = x[x <= z_krit]
ax.fill_between(x_tip2, stats.norm.pdf(x_tip2, mu_a, sigma),
                color='mediumpurple', alpha=0.6,
                label=f'Tip 2 Hata (β = {stats.norm.cdf(z_krit, mu_a, sigma):.3f})')
ax.axvline(z_krit, color='black', linestyle='--', linewidth=1.5)
ax.set_title('Tip 2 Hata\nHₐ doğruyken H₀\'ı reddetmemek')
ax.set_xlabel('Test istatistiği')
ax.legend(fontsize=9)
ax.set_yticks([])

plt.suptitle('Karar Hataları: Tip 1 ve Tip 2', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# α ve β arasındaki denge: α küçüldükçe β büyür
alpha_degerler = np.linspace(0.001, 0.20, 200)
mu_0_val, mu_a_val, sigma_val = 0, 2, 1

beta_degerler = []
for a in alpha_degerler:
    z_c = stats.norm.ppf(1 - a, mu_0_val, sigma_val)
    beta = stats.norm.cdf(z_c, mu_a_val, sigma_val)
    beta_degerler.append(beta)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alpha_degerler, beta_degerler, 'tomato', linewidth=2.5, label='Tip 2 Hata (β)')
ax.plot(alpha_degerler, 1 - np.array(beta_degerler), 'steelblue',
        linewidth=2.5, label='Güç = 1 - β')
ax.axvline(0.05, color='gray', linestyle='--', label='α = 0.05 (yaygın eşik)')
ax.set_xlabel('Anlamlılık Düzeyi (α = Tip 1 Hata Oranı)')
ax.set_ylabel('Olasılık')
ax.set_title('α Küçüldükçe Tip 2 Hata Olasılığı (β) Artar')
ax.legend(fontsize=10)
ax.set_xlim(0, 0.20)
plt.tight_layout()
plt.show()
print("α'yı küçültmek Tip 1 hatayı azaltır ancak Tip 2 hatayı artırır. Denge kurulmalıdır!")

---
## 7. Kapsamlı Uygulama: Adım Adım Hipotez Testi

Sunumdaki cinsiyete dayalı ayrımcılık deneyinden başlayarak, tüm adımları tek bir fonksiyonla özetleyelim.

In [ ]:
def tam_hipotez_testi(p_hat, n, p_0=0.50, guven=0.95, yon='iki_yonlu', alpha=0.05):
    """
    Bir oran için tam istatistiksel çıkarım:
    1) MLT koşul kontrolü
    2) Güven aralığı
    3) Hipotez testi
    """
    print("\n" + "#" * 60)
    print("  ADIM 1: MLT KOŞUL KONTROLÜ")
    print("#" * 60)
    mlt_kosul_kontrol(n=n, p_hat=p_hat)

    print("\n" + "#" * 60)
    print(f"  ADIM 2: %{int(guven*100)} GÜVEN ARALIĞI")
    print("#" * 60)
    guven_araligi(p_hat=p_hat, n=n, guven=guven)

    print("\n" + "#" * 60)
    print("  ADIM 3: HİPOTEZ TESTİ")
    print("#" * 60)
    hipotez_testi_oran(p_hat=p_hat, n=n, p_0=p_0, yon=yon, alpha=alpha)

# Facebook ilgi kategorileri: %41 rahat karşıladı, n=850
# H₀: p=0.50  Hₐ: p≠0.50
tam_hipotez_testi(p_hat=0.41, n=850, p_0=0.50, yon='iki_yonlu')

---
## 8. Özet ve Kavram Haritası

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║        BÖLÜM 5 – İSTATİSTİKSEL ÇIKARIM ÖZETİ                   ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  NOKTA TAHMİNİ                                                   ║
║    p̂ = örneklem oranı → p (kitle oranı) için en iyi tahmin      ║
║                                                                  ║
║  ÖRNEKLEME DAĞILIMI (MLT)                                        ║
║    p̂ ~ N(p, SE)    SE = √(p(1-p)/n)                             ║
║    Koşullar: bağımsızlık + np ≥ 10 + n(1-p) ≥ 10                ║
║                                                                  ║
║  GÜVEN ARALIĞI                                                   ║
║    p̂ ± z* × SE                                                  ║
║    %90: z*=1.645  |  %95: z*=1.960                              ║
║    %98: z*=2.326  |  %99: z*=2.576                              ║
║                                                                  ║
║  HİPOTEZ TESTİ                                                   ║
║    Z = (p̂ - p₀) / SE₀                                           ║
║    p-değeri < α → H₀ reddedilir                                  ║
║    p-değeri ≥ α → H₀ reddedilemez                                ║
║                                                                  ║
║  KARAR HATALARI                                                  ║
║    Tip 1 (α): H₀ doğruyken reddetmek                            ║
║    Tip 2 (β): Hₐ doğruyken reddetmemek                          ║
║    α ↓ → β ↑  (ters orantılı denge)                             ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## 9. Alıştırmalar

### Alıştırma 1
Bir anket, 500 kişilik örneklemde %62'sinin bir ürünü memnuniyetle kullandığını gösteriyor. 
- (a) MLT koşullarını kontrol edin.
- (b) %95 güven aralığını hesaplayın.
- (c) Ürün memnuniyetinin %50'den fazla olduğunu test edin (α=0.05).

### Alıştırma 2
Gerçek kitle oranı p=0.3 için, n=20, 50, 100, 500 değerlerinde örnekleme dağılımlarını çizin ve MLT koşulunu görsel olarak yorumlayın.

### Alıştırma 3
Aşağıdaki senaryolarda hangi hata türü daha tehlikelidir? Neden?
- (a) Kanser taraması testi
- (b) Spam e-posta filtresi
- (c) Uçak parçası kalite kontrolü

In [ ]:
# Alıştırma 1 – Çözüm alanı
# (a) MLT koşulları
mlt_kosul_kontrol(n=500, p_hat=0.62)

# (b) Güven aralığı
guven_araligi(p_hat=0.62, n=500, guven=0.95)

# (c) Hipotez testi: H₀: p=0.50, Hₐ: p>0.50 (tek yönlü)
hipotez_testi_oran(p_hat=0.62, n=500, p_0=0.50, yon='sag', alpha=0.05)